# Chapter 6 — IAAIS Classifier

This notebook demonstrates supervised exercise-window classification and its Knowledge Base and Uncertainty Module connections. All labels and measurements used below are synthetic teaching examples. They do not measure real-world performance or calibration.

The current task is one label per accelerometer/gyroscope window (for example, rest or an exercise class). It does not estimate repetitions, sets, form, diagnosis, or injury.

## 1. Engineer features from a sensor window

The first feature set summarizes each axis and the 3D magnitudes, and records sampling information. This simple baseline is orientation-sensitive and does not remove gravity or filter noise.

In [ ]:
import numpy as np
from iaais.classifier import (
    Classifier, InertialFeatureEngineer, TrainingExample,
    write_sensor_features_to_knowledge_base,
)
from iaais.knowledge_base import FactStatus, KnowledgeBase
from iaais.uncertainty import UncertaintyModule

accelerometer = np.array([[0.1, 0.0, 9.8], [0.3, 0.2, 9.7], [-0.2, 0.1, 9.9], [0.2, -0.1, 9.8]])
gyroscope = np.array([[0.0, 0.1, 0.0], [0.2, 0.0, 0.1], [-0.1, 0.2, 0.0], [0.1, -0.1, 0.2]])
engineered = InertialFeatureEngineer().transform(
    accelerometer, gyroscope, sample_rate_hz=50, sensor_profile="wrist"
)
len(engineered), engineered["accelerometer_magnitude_rms"], engineered["sensor_profile"]

## 2. Write feature facts and labeled examples to the Knowledge Base

Features use the predicate sensor_feature(window_id, feature_name, value). The labels below are confirmed review labels. The deliberately imbalanced, tiny toy set makes the example easy to run; it is not suitable for deployment.

In [ ]:
training_kb = KnowledgeBase()
toy_windows = [
    ("rest-1", 0.05, "rest"), ("rest-2", 0.08, "rest"),
    ("rest-3", 0.11, "rest"), ("rest-4", 0.15, "rest"),
    ("rest-5", 0.18, "rest"), ("squat-1", 0.80, "squat"),
    ("squat-2", 0.95, "squat"), ("squat-3", 1.10, "squat"),
    ("press-1", 1.80, "press"), ("press-2", 2.10, "press"),
]
training_groups = {}
for window_id, energy, label in toy_windows:
    write_sensor_features_to_knowledge_base(
        training_kb, window_id,
        {"movement_energy": energy, "sensor_profile": "wrist"},
        source="synthetic_toy_example",
    )
    training_kb.assert_fact(
        "exercise_label", (window_id, label),
        status=FactStatus.CONFIRMED, source="synthetic_toy_label",
    )
    training_groups[window_id] = f"synthetic-session-{window_id}"

classifier = Classifier().fit_from_knowledge_base(
    training_kb, [row[0] for row in toy_windows], group_ids=training_groups
)
classifier.classes_

## 3. Evaluate on separate toy recording groups

Macro-F1 is the primary metric because it gives each class equal weight. Balanced accuracy, per-class precision/recall, and the confusion matrix provide context. The evaluator rejects group IDs also used in training to catch a common source of window leakage.

In [ ]:
holdout = [
    TrainingExample({"movement_energy": 0.10, "sensor_profile": "wrist"}, "rest", group_id="heldout-rest-session"),
    TrainingExample({"movement_energy": 0.92, "sensor_profile": "wrist"}, "squat", group_id="heldout-squat-session"),
    TrainingExample({"movement_energy": 2.00, "sensor_profile": "wrist"}, "press", group_id="heldout-press-session"),
]
evaluation = classifier.evaluate(holdout)
{
    "primary_metric": evaluation.primary_metric,
    "macro_f1": evaluation.macro_f1,
    "balanced_accuracy": evaluation.balanced_accuracy,
    "per_class_precision": dict(evaluation.per_class_precision),
    "per_class_recall": dict(evaluation.per_class_recall),
    "confusion_matrix": evaluation.confusion_matrix,
}

## 4. Predict, preserve the full distribution, and request review

The classifier records the top label and one fact per class probability. All prediction facts stay proposed for review. The 0.60 threshold is only an initial review policy. Logistic-regression probabilities are not automatically calibrated; calibration needs a sufficiently large, representative labeled holdout set.

In [ ]:
prediction_kb = KnowledgeBase()
write_sensor_features_to_knowledge_base(
    prediction_kb, "new-window-1",
    {"movement_energy": 0.93, "sensor_profile": "wrist"},
    source="synthetic_toy_example", provenance=("synthetic-measurement-1",),
)
uncertainty = UncertaintyModule()
prediction = classifier.predict_from_knowledge_base(
    prediction_kb, "new-window-1", uncertainty_module=uncertainty
)
{
    "label": prediction.label,
    "probabilities": dict(prediction.probabilities.probabilities),
    "review_required": prediction.review_required,
    "feature_contributions": [
        (row.feature_name, row.contribution)
        for row in prediction.feature_contributions
    ],
    "proposed_kb_fact": prediction.prediction_fact,
    "probability_fact_count": len(prediction.probability_facts),
    "uncertainty_posterior": dict(prediction.uncertainty_report.posterior.probabilities),
}

## What this implementation includes

- Numeric and categorical feature vectors, with validation and KB feature provenance.
- A regularized logistic-regression baseline with balanced class weights.
- Macro-F1 and supporting holdout metrics, with an optional recording-group leakage check.
- Full class probabilities, a model-score contribution explanation, proposed KB facts, and an optional full-distribution Uncertainty Module update.

It does not include a real labeled dataset, model artifact persistence, automatic feature selection, probability calibration, out-of-distribution detection, or an Expert Module implementation. The Expert Module can consume the proposed prediction and per-class probability facts after its own rules are added.